# Week 8 개념 정리: LangChain 프레임워크 기초

이 노트북은 4일치 학습 노트를 위한 실행 가능한 동반 자료입니다. Day 2 섹션에서 실제 API 키가 필요해 스키마로만 보여주는, 명확히 표시된 셀 하나를 제외하면 모든 코드 셀은 이 환경에서 실제로 실행되었습니다 -- `pip install langchain-core`로 설치한 `langchain-core` 0.3.86을 Python 3.9.6에서 실행했습니다. 아래 출력은 각 셀을 실제로 실행해서 얻은 진짜 결과이며, 옮겨 적거나 예측한 값이 아닙니다.

In [ ]:
# 아래 모든 셀의 출력을 지저분하게 만드는, 무해하지만 시끄러운
# urllib3/LibreSSL 경고를 숨긴다.
import warnings
warnings.filterwarnings("ignore")

import langchain_core
print("langchain_core version:", langchain_core.__version__)


langchain_core version: 0.3.86


## Day 1: LangChain과 LCEL 파이프

`Runnable.__or__`는 두 Runnable을 하나의 `RunnableSequence`로 합칩니다: 시퀀스에 대해 `.invoke()`를 호출하면 첫 번째 단계를 실행한 뒤 그 출력을 두 번째 단계의 입력으로 넘깁니다. 아래에서는 이 메커니즘을 아무것도 숨기지 않고 작은 클래스(`MiniRunnable`)로 직접 재현해본 다음, 같은 조합 패턴을 실제로 설치된 `langchain_core`로 실행해봅니다.

In [ ]:
class MiniRunnable:
    """langchain_core.runnables.Runnable의 대역: 이번 레슨의 핵심인 하나의
    메커니즘만 남긴다 -- __or__가 여전히 .invoke()를 노출하는 시퀀스를
    만든다는 것."""

    def invoke(self, value):
        raise NotImplementedError

    def __or__(self, other):
        # a | b -> MiniSequence(a, b): 이 이상의 마법은 없다.
        return MiniSequence(self, other)


class MiniSequence(MiniRunnable):
    def __init__(self, first, second):
        self.first = first
        self.second = second

    def invoke(self, value):
        # `prompt | llm | parser`의 뒤에 숨어 있는 모든 트릭: 첫 번째
        # 단계의 반환값이 두 번째 단계의 인자가 된다.
        first_output = self.first.invoke(value)   # 1단계가 완전히 실행됨
        return self.second.invoke(first_output)     # 그 출력이 2단계로 흘러들어감


class FuncStep(MiniRunnable):
    """평범한 함수를 파이프라인 단계로 감싼다 -- 실제 LangChain은 호출
    가능한 객체를 파이프에 연결하면 이 변환을 (RunnableLambda로) 자동으로
    해주기 때문에, 이 래퍼를 직접 쓸 일은 거의 없다."""

    def __init__(self, fn):
        self.fn = fn

    def invoke(self, value):
        return self.fn(value)


class TinyTemplate(MiniRunnable):
    """PromptTemplate.from_template(...)의 로컬 대역."""

    def __init__(self, template):
        self.template = template

    def invoke(self, variables):
        # dict -> str : str.format과 같은 방식으로 {변수}를 채운다
        return self.template.format(**variables)


class StubForecaster(MiniRunnable):
    """모의 '모델': 실제 채팅 모델이 가질 법한 .invoke() 형태와 일치하지만
    스크립트로 정해진 문자열을 반환한다 -- 네트워크 호출 없음, 완전히
    결정론적, 배관이 제대로 되어 있는지 증명하기엔 충분하다."""

    def invoke(self, prompt_text):
        return f"[stub-forecast] based on '{prompt_text[:40]}...', expect steady demand"


weather_prompt = TinyTemplate("Given these signals: {signals}, forecast next week's sales.")
trim_and_shout = FuncStep(lambda text: text.strip().upper())

# dict -> str (채워진 템플릿) -> str (예측) -> str (대문자화)
forecast_chain = weather_prompt | StubForecaster() | trim_and_shout
print(type(forecast_chain).__name__)  # -> MiniSequence, 그 자체도 MiniRunnable

print(forecast_chain.invoke({"signals": "rising foot traffic, no promotions running"}))
print(forecast_chain.invoke({"signals": "holiday week, heavy discounting"}))


MiniSequence
[STUB-FORECAST] BASED ON 'GIVEN THESE SIGNALS: RISING FOOT TRAFFIC...', EXPECT STEADY DEMAND
[STUB-FORECAST] BASED ON 'GIVEN THESE SIGNALS: HOLIDAY WEEK, HEAVY...', EXPECT STEADY DEMAND


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import Runnable

class ScriptedModel(Runnable):
    """채팅 모델의 대역: 실제 모델과 동일한 .invoke() 형태를 갖지만
    네트워크 호출은 전혀 없고, 실제 API 호출 한 번 쓰기 전에 배관을
    테스트할 수 있도록 완전히 결정론적인 출력을 낸다."""
    def invoke(self, prompt_value, config=None, **kwargs):
        # prompt_value: StringPromptValue (PromptTemplate의 출력 타입)
        text = prompt_value.to_string()  # StringPromptValue -> str
        return f"MOCK-REPLY: read {len(text)} chars starting '{text[:24]}'"

def to_upper(text: str) -> str:
    return text.upper()

prompt = PromptTemplate.from_template("Explain {topic} in one short sentence.")
pv = prompt.invoke({"topic": "binary search"})
print("prompt.invoke ->", type(pv).__name__, ":", pv)

# 여기서 RunnableLambda 자동 변환이 일어난다: to_upper는 직접 만든
# Runnable이 아니라 평범한 함수지만, 파이프에 연결해도 그대로 동작한다.
chain = prompt | ScriptedModel() | to_upper
print("chain type:", type(chain).__name__)  # -> RunnableSequence

# dict -> StringPromptValue -> str -> str
result = chain.invoke({"topic": "binary search"})
print("result:", result)


prompt.invoke -> StringPromptValue : text='Explain binary search in one short sentence.'
chain type: RunnableSequence
result: MOCK-REPLY: READ 44 CHARS STARTING 'EXPLAIN BINARY SEARCH IN'


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

class FakeChatModel(Runnable):
    """실제 채팅 모델의 .invoke() 계약을 그대로 따른다: ChatPromptValue
    (또는 list[BaseMessage])를 받아 AIMessage를 반환한다."""
    def invoke(self, input_value, config=None, **kwargs):
        # ChatPromptValue -> list[BaseMessage], len=2 (system, human)
        messages = input_value.to_messages()
        last_human = next(
            (m.content for m in reversed(messages) if isinstance(m, HumanMessage)),
            "",
        )
        return AIMessage(content=f"[fake-llm] you said: {last_human}")

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a terse assistant."),
    ("human", "{question}"),
])

# 각 중간 형태를 .invoke() 한 번씩 실행하며 명시적으로 확인한다:
pv = chat_prompt.invoke({"question": "What is LCEL?"})
print("ChatPromptValue.to_messages():", pv.to_messages())

msg = FakeChatModel().invoke(pv)
print("model output:", type(msg).__name__, "content=", msg.content)

parsed = StrOutputParser().invoke(msg)
print("parser output:", type(parsed).__name__, ":", parsed)

# 이제 같은 세 단계를 하나로 합쳐진 체인으로:
# dict -> ChatPromptValue -> AIMessage -> str
chain = chat_prompt | FakeChatModel() | StrOutputParser()
result = chain.invoke({"question": "What is LCEL?"})
print("chain result:", type(result).__name__, ":", result)


ChatPromptValue.to_messages(): [SystemMessage(content='You are a terse assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is LCEL?', additional_kwargs={}, response_metadata={})]
model output: AIMessage content= [fake-llm] you said: What is LCEL?
parser output: str : [fake-llm] you said: What is LCEL?
chain result: str : [fake-llm] you said: What is LCEL?


In [ ]:
import time
from langchain_core.runnables import RunnableLambda

def slow_double(x):
    time.sleep(0.2)  # 모델 API에 대한 느린 네트워크 호출을 흉내낸다
    return x * 2

r = RunnableLambda(slow_double)
t0 = time.time()
out = r.batch([1, 2, 3, 4])   # 스레드 풀을 통해 4개 호출을 동시에 실행한다
elapsed = time.time() - t0
print("results:", out)
print(f"elapsed: {elapsed:.2f}s (순차 호출 4번이었다면 약 0.8s가 걸렸을 것)")


results: [2, 4, 6, 8]
elapsed: 0.21s (4 sequential calls would take roughly 0.8s)


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# RunnableParallel은 여러 Runnable을 *같은* 입력에 대해 실행하고 그
# 결과를 dict로 모은다. RunnablePassthrough는 입력을 그대로 반환할
# 뿐인데, 이것이 변환된 값과 함께 원본 값을 그대로 이어서 전달하는
# 방법이다 (많은 검색 증강 체인의 뼈대가 되는 패턴).
branch = RunnableParallel(
    upper=RunnableLambda(lambda x: x.upper()),
    length=RunnableLambda(lambda x: len(x)),
    original=RunnablePassthrough(),
)
print(branch.invoke("hello world"))


{'upper': 'HELLO WORLD', 'length': 11, 'original': 'hello world'}


## Day 2: 재사용 가능한 템플릿, 교체 가능한 모델, 수동 파싱

템플릿은 한 번 정의되고 `.batch()`를 통해 여러 다른 입력으로 재사용됩니다. 모델 제공자를 교체하는 일은 대부분 생성자 한 줄만 바꾸는 작업인데, 모든 채팅 모델 클래스가 같은 `Runnable` 계약을 구현하기 때문입니다(아래는 실제로 실행하려면 실제 설치와 API 키가 필요하므로 스키마로만 보여줍니다). 이어서: 원본 모델 출력을 직접 파싱하고 검증하는 방법, 그리고 `JsonOutputParser`와 실제로 직접 비교하며 프레임워크 파서가 정확히 무엇을 검사하고 무엇을 검사하지 않는지 확인합니다.

In [ ]:
review_prompt = PromptTemplate.from_template(
    "Rate the sentiment of this review from 1-5 and give one reason: {review}"
)
reviews = [
    {"review": "The battery died after two days."},
    {"review": "Fast shipping, exactly as described."},
]
# .batch()는 템플릿을 입력 목록 전체에 대해 실행한다 -- Python for문이
# 아니라 LCEL다운 방식이다. 같은 템플릿 객체, 다른 {review} 값.
for pv in review_prompt.batch(reviews):
    print(pv.to_string())

# .partial()은 고정 변수를 채워 넣고 *새로운* PromptTemplate을 반환한다;
# base 자체는 손대지 않은 채로 남는다.
base = PromptTemplate.from_template("You are a {persona}. Answer: {question}")
support_bot = base.partial(persona="support agent")
print(support_bot.invoke({"question": "how do I reset my password?"}).to_string())
print("base is untouched:", base.input_variables)  # -> 여전히 'persona'도 필요함


Rate the sentiment of this review from 1-5 and give one reason: The battery died after two days.
Rate the sentiment of this review from 1-5 and give one reason: Fast shipping, exactly as described.
You are a support agent. Answer: how do I reset my password?
base is untouched: ['persona', 'question']


아래 셀은 이 샌드박스에서 실행할 수 없습니다 -- `langchain-openai`와 `langchain-anthropic` 설치, 그리고 두 제공자 모두에 대한 실제 API 키가 필요합니다. 실제로 사용할 수 있는, 문법적으로 올바른 코드로 포함되어 있습니다; 노트북 전체가 네트워크 호출이나 키 없이도 순서대로 실행되도록 모든 줄이 주석 처리되어 있습니다.

In [ ]:
# 이 샌드박스에서는 실행되지 않음 -- 둘 다 실제 설치 + API 키가 필요.
# from langchain_openai import ChatOpenAI
# from langchain_anthropic import ChatAnthropic
#
# model_a = ChatOpenAI(model="gpt-4o-mini")
# model_b = ChatAnthropic(model="claude-3-5-haiku-20241022")
# for model in (model_a, model_b):
#     response = model.invoke("Summarize LCEL in one sentence.")
#     print(response.content)  # 둘 다 AIMessage를 반환 -- .content를 읽는 방식도 동일
print("skipped: requires langchain-openai / langchain-anthropic + real API keys")


skipped: requires langchain-openai / langchain-anthropic + real API keys


In [ ]:
import json

def parse_and_validate(raw: str) -> dict:
    """수작업으로 짠 파싱 + 검증 -- 프레임워크 출력 파서를 쓰지 않는다."""
    data = json.loads(raw)  # 형식이 잘못된 JSON이면 json.JSONDecodeError 발생

    required = {"label", "confidence", "tags"}
    missing = required - data.keys()
    if missing:
        raise ValueError(f"missing keys: {missing}")

    if data["label"] not in {"positive", "neutral", "negative"}:
        raise ValueError(f"unexpected label: {data['label']}")

    conf = data["confidence"]
    if not isinstance(conf, (int, float)) or isinstance(conf, bool) or not (0.0 <= conf <= 1.0):
        raise ValueError(f"confidence out of range: {conf}")

    if not isinstance(data["tags"], list):
        raise ValueError("tags must be a list")

    return data  # -> dict, 네 가지 불변조건 모두 확인 완료, 이후 안전하게 사용 가능

good = '{"label": "positive", "confidence": 0.82, "tags": ["shipping", "praise"]}'
print(parse_and_validate(good))

# "amazing"은 형식은 올바른 JSON이지만 이 시스템이 이해하는 세 가지
# 레이블 중 하나가 아니다 -- parse_and_validate가 올바르게 거부한다.
bad = '{"label": "amazing", "confidence": 0.82, "tags": ["shipping"]}'
try:
    parse_and_validate(bad)
except ValueError as e:
    print("correctly rejected:", e)


{'label': 'positive', 'confidence': 0.82, 'tags': ['shipping', 'praise']}
correctly rejected: unexpected label: amazing


In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

# 닫는 중괄호가 없이 잘린 객체 -- json.loads라면 그냥 거부했겠지만,
# JsonOutputParser는 부분 스트리밍 JSON을 위해 만들어져서 복구해낸다.
print("recovered from truncation:",
      parser.invoke('{"priority": "high", "category": "billing", "eta_minutes": 15'))

# 문법적으로는 유효하지만 의미적으로는 틀림 ("urgent"는 실제 priority가
# 아니고, eta_minutes는 int가 아니라 문자열) -- 오류 없이 그대로 통과.
print("semantically bad, no error:",
      parser.invoke('{"priority": "urgent", "eta_minutes": "soon"}'))

# 진짜로 JSON이 아닌 텍스트 -- 이 경우에만 실제로 예외가 발생한다.
try:
    parser.invoke("Sure! Here is the answer: not json at all")
except Exception as e:
    print("correctly raised:", type(e).__name__)


recovered from truncation: {'priority': 'high', 'category': 'billing', 'eta_minutes': 15}
semantically bad, no error: {'priority': 'urgent', 'eta_minutes': 'soon'}
correctly raised: OutputParserException


## Day 3: 안전한 계산기 도구와 FAQ 도구

`eval()`은 사용자가 제공한 표현식에 대해 안전하지 않습니다. `ast.literal_eval` 역시 계산기로 쓰기에는 부족합니다: 리터럴 상수와 컨테이너만 파싱할 뿐, 두 리터럴 사이의 산술 연산은 파싱하지 못합니다 -- 아래에서 실제로 확인하듯 `ast.literal_eval("12 * 8")`은 정말로 `ValueError`를 발생시킵니다. 해법은 산술 노드만 화이트리스트에 올리고 나머지(이름, 함수 호출, 속성 접근, 서브스크립트 등)는 모두 거부하는 커스텀 AST 워커이며, 아래에서 실제 공격 문자열로도 검증합니다.

In [ ]:
import ast

print(ast.literal_eval("12"))          # -> 12          (리터럴: 문제없음)
print(ast.literal_eval("[1, 2, 3]"))   # -> [1, 2, 3]    (리터럴 컨테이너: 문제없음)

try:
    ast.literal_eval("12 * 8")  # BinOp(곱셈)은 애초에 literal_eval의
                                 # 제한된 문법에 포함되어 있지 않다
except ValueError as e:
    print("literal_eval('12 * 8') raises:", e)


12
[1, 2, 3]
literal_eval('12 * 8') raises: malformed node or string: <ast.BinOp object at 0x10a374490>


In [ ]:
import operator

# 화이트리스트: 이 계산기가 지원하는 산술 연산, 그 이상은 없다.
_BIN_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
}
_UNARY_OPS = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _eval(node):
    # ast.parse(..., mode="eval")은 항상 실제 표현식을 Expression 노드로
    # 감싼다 -- 한 번 벗겨내고 .body를 재귀 호출한다.
    if isinstance(node, ast.Expression):
        return _eval(node.body)

    # bool은 명시적으로 제외한다: Python에서 isinstance(True, int)는
    # True이므로, 이 검사가 없으면 "True * 8"이 조용히 8로 평가된다.
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
            and not isinstance(node.value, bool):
        return node.value  # -> int | float

    if isinstance(node, ast.BinOp) and type(node.op) in _BIN_OPS:
        left = _eval(node.left)    # 재귀: 왼쪽도 그 자체로 BinOp일 수 있다
        right = _eval(node.right)  # 재귀: 오른쪽도 그 자체로 BinOp일 수 있다
        return _BIN_OPS[type(node.op)](left, right)

    if isinstance(node, ast.UnaryOp) and type(node.op) in _UNARY_OPS:
        return _UNARY_OPS[type(node.op)](_eval(node.operand))

    # Name, Call, Attribute, Subscript, List, Compare, ... 는 모두 여기까지
    # 흘러와서 거부된다.
    raise ValueError(f"disallowed expression: {type(node).__name__}")

def calc(expr: str):
    tree = ast.parse(expr, mode="eval")  # str -> ast.Expression
    return _eval(tree)                    # ast.Expression -> int | float

for expr in ["12 * 8", "(9 - 3) ** 2 / 4", "-8 + 20 / 4", "100 % 7", "2 ** 10"]:
    print(f"{expr:20} => {calc(expr)}")

print()
# 실제 공격 문자열 -- 하나하나가 "위험해 보인다"는 패턴 매칭이 아니라
# 이름으로 거부된다: Attribute/Call/List/Name은 애초에 위 화이트리스트에
# 추가된 적이 없을 뿐이다.
for bad in ["__import__('os').system('echo pwned')", "(1).__class__.__bases__",
            "open('secrets.txt')", "[1, 2, 3]", "a + 1"]:
    try:
        calc(bad)
        print("SHOULD NOT REACH:", bad)
    except ValueError as e:
        print(f"rejected: {bad!r:45} -> {e}")

print()
try:
    calc("1/0")
except ZeroDivisionError as e:
    # calc()는 Div 노드를 정확히 평가한다; 실패한 것은 나눗셈 그 자체가
    # 런타임에 일으킨 것이지 disallowed-expression 거부가 아니다 -- 실제
    # 호출부라면 ValueError와 함께 이것도 잡아야 한다.
    print("division by zero surfaces as:", type(e).__name__, "->", e)


12 * 8               => 96
(9 - 3) ** 2 / 4     => 9.0
-8 + 20 / 4          => -3.0
100 % 7              => 2
2 ** 10              => 1024

rejected: "__import__('os').system('echo pwned')"       -> disallowed expression: Call
rejected: '(1).__class__.__bases__'                     -> disallowed expression: Attribute
rejected: "open('secrets.txt')"                         -> disallowed expression: Call
rejected: '[1, 2, 3]'                                   -> disallowed expression: List
rejected: 'a + 1'                                       -> disallowed expression: Name

division by zero surfaces as: ZeroDivisionError -> division by zero


In [ ]:
_FAQ = [
    (("refund", "money back"), "Refunds post within 5-7 business days after we receive the return."),
    (("hours", "open"), "Support is staffed 9am-6pm, Monday through Friday."),
    (("shipping", "delivery"), "Standard shipping takes 3-5 business days."),
]

def faq(question: str):
    q = question.lower()  # 대소문자를 구분하지 않는 키워드 매칭
    for keywords, answer in _FAQ:
        if any(kw in q for kw in keywords):
            return answer  # -> str: 목록 순서상 처음 일치한 답변
    return None  # -> None: 어떤 키워드도 일치하지 않음

def handle(user_input: str) -> str:
    has_digit = any(c.isdigit() for c in user_input)
    has_operator = any(op in user_input for op in "+-*/")
    if has_digit and has_operator:
        try:
            return f"= {calc(user_input)}"
        except (ValueError, ZeroDivisionError) as e:
            return f"couldn't evaluate that: {e}"
    answer = faq(user_input)
    return answer if answer else "no matching tool for that yet."

# 마지막 입력은 라우터의 잘 알려진 사각지대다: 숫자도, 연산자 기호도,
# FAQ 키워드도 없다 -- 사람이 보면 곧바로 산술 질문임을 알 수 있는데도.
for text in ["9 * 6", "can I renew my loan?", "what are your hours",
             "what is nine times six"]:
    print(f"{text!r:30} -> {handle(text)}")


'9 * 6'                        -> = 54
'can I renew my loan?'         -> no matching tool for that yet.
'what are your hours'          -> Support is staffed 9am-6pm, Monday through Friday.
'what is nine times six'       -> no matching tool for that yet.


## Day 4: 메모리, 성장, 그리고 영속성

모델 호출은 하나하나 상태가 없습니다 -- "메모리"는 이전 턴들을 다음 프롬프트의 일부로 다시 보내는 것을 뜻하며, 그래서 경계 없는 대화는 이후의 모든 프롬프트를 점점 더 크게(그리고 더 느리고, 더 비싸게) 만듭니다. 이는 아래에서 실제로 측정합니다. 두 가지 경계 설정 전략: 최근 윈도우를 유지하거나, 오래된 턴을 주기적으로 요약에 접어 넣는 것. 이력을 영속화하면(예: SQLite에) 대화가 재시작을 견뎌낼 수 있지만 -- 사용자가 입력한 것은 PII를 포함해 적극적으로 마스킹하지 않는 한 그대로 저장되며, 이는 데이터 보존과 컴플라이언스 측면에서 중요합니다. 실제로 (불완전하게) 동작하는 정규식 마스킹을 직접 돌려보면 이 한계가 구체적으로 드러납니다.

In [ ]:
def build_prompt(history, new_message):
    convo = "\n".join(f"{role}: {text}" for role, text in history)
    return f"{convo}\nuser: {new_message}\nassistant:"

history = []
sizes = []
for turn in range(1, 21):
    user_msg = f"question {turn} about the order"
    prompt = build_prompt(history, user_msg)  # 지금까지의 모든 것을 매번 다시 직렬화
    sizes.append(len(prompt))
    history.append(("user", user_msg))
    history.append(("assistant", f"answer {turn}"))

print("turn 1 prompt length:", sizes[0], "chars")
print("turn 20 prompt length:", sizes[-1], "chars")
print("growth factor:", round(sizes[-1] / sizes[0], 1), "x over 20 turns")
print("full history length:", len(history), "entries")


turn 1 prompt length: 44 chars
turn 20 prompt length: 1071 chars
growth factor: 24.3 x over 20 turns
full history length: 40 entries


In [ ]:
def windowed_history(history, max_turns=6):
    """완화책 1: 가장 최근 N턴만 유지하고 나머지는 버린다."""
    return history[-max_turns:]  # -> list, len <= max_turns

def compact_history(history, keep_recent=6, summarize=None):
    """완화책 2: 오래된 턴들을 하나의 누적 요약 한 줄로 압축한다."""
    if len(history) <= keep_recent:
        return history  # 아직 압축할 게 없음
    old, recent = history[:-keep_recent], history[-keep_recent:]
    old_text = "\n".join(f"{r}: {t}" for r, t in old)
    # 실제 시스템에서 `summarize`는 보통 또 다른 LLM 호출이다 -- 여기서는
    # 별도 모델 없이 이 셀을 실행할 수 있도록 기본값으로 거친 절삭을 쓴다.
    summary = summarize(old_text) if summarize else old_text[:200] + "..."
    return [("system", f"earlier conversation summary: {summary}")] + list(recent)

print("full history:", len(history), "entries")
print("windowed turns kept:", len(windowed_history(history)))
print("compacted turns kept:", len(compact_history(history)))


full history: 40 entries
windowed turns kept: 6
compacted turns kept: 7


In [ ]:
import sqlite3
import os

def save_history_sqlite(history, db_path, conversation_id="demo"):
    """프로세스가 재시작되어도 살아남도록 이력을 영속화한다."""
    conn = sqlite3.connect(db_path)
    conn.execute(
        "CREATE TABLE IF NOT EXISTS turns "
        "(conversation_id TEXT, turn_index INTEGER, speaker TEXT, text TEXT)"
    )
    conn.execute("DELETE FROM turns WHERE conversation_id = ?", (conversation_id,))
    conn.executemany(
        "INSERT INTO turns VALUES (?, ?, ?, ?)",
        [(conversation_id, i, speaker, text) for i, (speaker, text) in enumerate(history)],
    )
    conn.commit()
    conn.close()

def load_history_sqlite(db_path, conversation_id="demo"):
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        "SELECT speaker, text FROM turns WHERE conversation_id = ? ORDER BY turn_index",
        (conversation_id,),
    ).fetchall()
    conn.close()
    return rows

db_path = "/tmp/week8_demo_conversations.db"
if os.path.exists(db_path):
    os.remove(db_path)  # 다시 실행해도 재현 가능하도록 깨끗하게 시작한다

save_history_sqlite(history[:4], db_path=db_path)
roundtrip = load_history_sqlite(db_path=db_path)
print(roundtrip)
print("roundtrip matches original:", roundtrip == history[:4])
os.remove(db_path)


[('user', 'question 1 about the order'), ('assistant', 'answer 1'), ('user', 'question 2 about the order'), ('assistant', 'answer 2')]
roundtrip matches original: True


In [ ]:
import re

_EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
_PHONE_RE = re.compile(r"(?<!\w)\+?\d[\d\-\s]{7,}\d(?!\w)")

def redact_pii(text: str) -> str:
    text = _EMAIL_RE.sub("[redacted-email]", text)
    text = _PHONE_RE.sub("[redacted-phone]", text)
    return text

samples = [
    "my email is a.kim@example.com, call me at 555-123-4567 too",
    "reach me at +1 415 555 0199 or backup jane.doe+work@corp.co.kr",
    # 말로 풀어쓴 숫자: 정규식은 오직 숫자 문자만 찾기 때문에 이 문장은
    # 완전히 그대로 통과한다 -- 실제로 검증된 진짜 빈틈이다.
    "my number is five five five, one two three, four five six seven",
]
for s in samples:
    print(redact_pii(s))


my email is [redacted-email], call me at [redacted-phone] too
reach me at [redacted-phone] or backup [redacted-email]
my number is five five five, one two three, four five six seven
